In [1]:
# =====================================================================
# CÉLULA 1 — Importações
# =====================================================================
import os
import re
import sys
import unicodedata
from datetime import date, datetime, time, timedelta
from pathlib import Path

import pandas as pd

ENCODINGS_TENTATIVA = ("utf-8-sig", "utf-8", "latin-1", "cp1252")
SEPARADORES_TENTATIVA = (";", ",", "\t", "|")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

In [2]:
# =====================================================================
# CÉLULA 2 — Caminhos dinâmicos
# =====================================================================
def get_base_dir() -> Path:
    if getattr(sys, "frozen", False):
        return Path(sys.executable).resolve().parent

    vsc = globals().get("__vsc_ipynb_file__")
    if vsc:
        return Path(vsc).resolve().parent

    arquivo = globals().get("__file__")
    if arquivo:
        return Path(arquivo).resolve().parent

    return Path.cwd()


BASE_DIR = get_base_dir()


def resolver_pasta_arquivos(nome: str = "arquivos") -> Path:
    """1. AFD_PASTA  2. <base>/arquivos  3. <_MEIPASS>/arquivos  4. cria"""
    env = os.getenv("AFD_PASTA")
    if env:
        return Path(env).expanduser().resolve()

    candidatos = [BASE_DIR / nome]
    meipass = getattr(sys, "_MEIPASS", None)
    if meipass:
        candidatos.append(Path(meipass) / nome)

    for candidato in candidatos:
        if candidato.is_dir():
            return candidato

    destino = BASE_DIR / nome
    destino.mkdir(parents=True, exist_ok=True)
    return destino


PASTA_ARQUIVOS = resolver_pasta_arquivos()

print("BASE_DIR       :", BASE_DIR)
print("PASTA_ARQUIVOS :", PASTA_ARQUIVOS)

BASE_DIR       : \\AGROSRV\Automacoes\Marcacao_Pontos
PASTA_ARQUIVOS : \\AGROSRV\Automacoes\Marcacao_Pontos\arquivos


In [3]:
# =====================================================================
# CÉLULA 3 — Arquivos CSV encontrados
# =====================================================================
EXTENSOES_CSV = {".csv", ".txt"}


def parece_relatorio_rhid(caminho: Path) -> bool:
    """Confere se a 1ª linha tem cara de cabeçalho do relatório do RHiD."""
    for enc in ENCODINGS_TENTATIVA:
        try:
            with caminho.open("r", encoding=enc, errors="strict") as f:
                cabecalho = f.readline().lower()
            break
        except UnicodeDecodeError:
            continue
    else:
        return False

    return ("cpf" in cabecalho or "funcionario" in cabecalho) and "dia" in cabecalho


def listar_relatorios(pasta: Path | str | None = None) -> list[Path]:
    pasta = Path(pasta or PASTA_ARQUIVOS)
    if not pasta.is_dir():
        return []
    return [
        item for item in sorted(pasta.iterdir())
        if item.is_file()
        and item.suffix.lower() in EXTENSOES_CSV
        and parece_relatorio_rhid(item)
    ]


ARQUIVOS = listar_relatorios()
print(f"{len(ARQUIVOS)} arquivo(s) em {PASTA_ARQUIVOS}")
for arq in ARQUIVOS:
    print(f"  - {arq.name}  ({arq.stat().st_size / 1024:.1f} KB)")

1 arquivo(s) em \\AGROSRV\Automacoes\Marcacao_Pontos\arquivos
  - relatorio_20261026_1501.CSV  (58.1 KB)


In [4]:
# =====================================================================
# CÉLULA 4 — Leiaute: resolução de colunas
# =====================================================================
CAMPOS_FIXOS = {
    "nome": ("nome_do_funcionario", "nome_funcionario", "funcionario", "nome", "colaborador"),
    "cpf": ("cpf_do_funcionario", "cpf_funcionario", "cpf", "cpf_colaborador"),
    "dia": ("dia", "data", "data_dia"),
    "previsto": ("previsto", "horario_previsto", "jornada_prevista", "escala", "horario_contratual"),
    "total_normais": ("total_normais", "normais", "total_normal"),
    "total_noturno": ("total_noturno", "noturno"),
    "dia_falta": ("dia_falta", "falta_dia"),
    "falta_e_atraso": ("falta_e_atraso", "faltas_e_atrasos", "falta_atraso"),
    "abono": ("abono", "abonos"),
    "extra_0_d": ("extra_0_d", "extra_0d"),
    "extra_60_d": ("extra_60_d", "extra_60d"),
    "extra_diurna": ("extra_diurna", "extras_diurna"),
    "extra_noturna": ("extra_noturna", "extras_noturna"),
    "banco_total": ("banco_total", "total_banco", "banco_dia"),
    "banco_saldo": ("banco_saldo", "saldo_banco", "saldo"),
    "justificativas": ("justificativas", "justificativa", "observacoes", "obs"),
}

CAMPOS_DURACAO = (
    "total_normais", "total_noturno", "dia_falta", "falta_e_atraso", "abono",
    "extra_0_d", "extra_60_d", "extra_diurna", "extra_noturna",
    "banco_total", "banco_saldo",
)

DIAS_ABREV = ("seg", "ter", "qua", "qui", "sex", "sáb", "dom")

# Intervalo NÃO remunerado após cada par de marcação:
#   após o par 1 (Saída 1 -> Entrada 2) = Café
#   após o par 2 (Saída 2 -> Entrada 3) = Almoço
ROTULO_INTERVALO = {1: "Café", 2: "Almoço", 3: "Intervalo 3", 4: "Intervalo 4"}


def normalizar_nome_coluna(nome) -> str:
    texto_ = str(nome or "").strip().lower()
    texto_ = unicodedata.normalize("NFKD", texto_)
    texto_ = "".join(c for c in texto_ if not unicodedata.combining(c))
    texto_ = re.sub(r"[^a-z0-9]+", "_", texto_)
    return texto_.strip("_")


def resolver_colunas(colunas) -> dict[str, str]:
    mapa_normalizado = {normalizar_nome_coluna(c): c for c in colunas}
    resolvido = {}

    for campo, candidatos in CAMPOS_FIXOS.items():
        for candidato in candidatos:
            if candidato in mapa_normalizado:
                resolvido[campo] = mapa_normalizado[candidato]
                break
        else:
            for candidato in candidatos:
                achado = next(
                    (real for norm, real in mapa_normalizado.items()
                     if norm.startswith(candidato)), None)
                if achado:
                    resolvido[campo] = achado
                    break

    return resolvido


def mapear_pares(colunas) -> dict[int, dict[str, str]]:
    pares: dict[int, dict[str, str]] = {}

    for coluna in colunas:
        norm = normalizar_nome_coluna(coluna)

        achado = re.match(r"^(entrada|ent)_?(\d+)$", norm)
        if achado:
            pares.setdefault(int(achado.group(2)), {})["entrada"] = coluna
            continue

        achado = re.match(r"^(saida|sai)_?(\d+)$", norm)
        if achado:
            pares.setdefault(int(achado.group(2)), {})["saida"] = coluna

    return dict(sorted(pares.items()))


def conferir_leiaute(df: pd.DataFrame) -> tuple[dict, dict, list[str]]:
    campos = resolver_colunas(df.columns)
    pares = mapear_pares(df.columns)

    problemas = []
    for obrigatorio in ("nome", "cpf", "dia"):
        if obrigatorio not in campos:
            problemas.append(
                f"coluna obrigatória não encontrada: {obrigatorio}")

    if not pares:
        problemas.append("nenhum par entrada/saída encontrado")

    return campos, pares, problemas

In [5]:
# =====================================================================
# CÉLULA 5 — Conversão de hora, duração, data, jornada prevista
# =====================================================================
VAZIOS = {"", "-", "--", "—", "–", "nan", "nat", "none", "null"}
RE_HORA = re.compile(r"(\d{1,2})\s*:\s*(\d{2})")
RE_DURACAO = re.compile(r"^(\d{1,3}):(\d{2})(?::(\d{2}))?$")
RE_INTERVALO = re.compile(r"(\d{1,2}:\d{2})\s*[-–]\s*(\d{1,2}:\d{2})")
RE_DIA_BR = re.compile(r"(\d{1,2})/(\d{1,2})(?:/(\d{2,4}))?")
RE_DIA_ISO = re.compile(r"(\d{4})-(\d{2})-(\d{2})")


def vazio(valor) -> bool:
    if valor is None:
        return True
    try:
        if pd.isna(valor):
            return True
    except (TypeError, ValueError):
        pass
    return str(valor).strip().lower() in VAZIOS


def texto(valor) -> str:
    return "" if vazio(valor) else str(valor).strip()


def somente_digitos(valor) -> str:
    return re.sub(r"\D", "", texto(valor))


def formatar_cpf(valor) -> str:
    d = somente_digitos(valor)
    return f"{d[:3]}.{d[3:6]}.{d[6:9]}-{d[9:]}" if len(d) == 11 else texto(valor)


def mascarar_cpf(valor) -> str:
    d = somente_digitos(valor)
    return f"{d[:3]}.***.***-{d[9:]}" if len(d) == 11 else "***"


def cpf_valido(valor) -> bool:
    d = somente_digitos(valor)
    if len(d) != 11 or len(set(d)) == 1:
        return False
    for tamanho in (9, 10):
        soma = sum(int(d[i]) * (tamanho + 1 - i) for i in range(tamanho))
        digito = 11 - soma % 11
        digito = 0 if digito >= 10 else digito
        if digito != int(d[tamanho]):
            return False
    return True


def parse_hora(valor) -> time | None:
    """'06:33', '06:33 (M)', '6:33' -> time. Ignora sufixo de marcação."""
    if vazio(valor):
        return None
    achado = RE_HORA.search(str(valor))
    if not achado:
        return None
    h, m = int(achado.group(1)), int(achado.group(2))
    if h > 23 or m > 59:
        return None
    return time(h, m)


def parse_duracao(valor) -> timedelta | None:
    """'04:30', '-05:17', '52:13' -> timedelta. Aceita negativo e > 24h."""
    if vazio(valor):
        return None
    bruto = str(valor).strip().replace(" ", "")
    negativo = bruto.startswith("-")
    bruto = bruto.lstrip("-")
    achado = RE_DURACAO.match(bruto)
    if not achado:
        return None
    td = timedelta(hours=int(achado.group(1)),
                   minutes=int(achado.group(2)),
                   seconds=int(achado.group(3) or 0))
    return -td if negativo else td


def parse_dia(valor, ano_padrao: int | None = None) -> date | None:
    """'26/09 SAB', '26/09/2026 SAB', '2026-09-26' -> date."""
    if vazio(valor):
        return None
    bruto = str(valor).strip()

    achado = RE_DIA_ISO.search(bruto)
    if achado:
        try:
            return date(int(achado.group(1)), int(achado.group(2)),
                        int(achado.group(3)))
        except ValueError:
            return None

    achado = RE_DIA_BR.search(bruto)
    if not achado:
        return None

    dia, mes = int(achado.group(1)), int(achado.group(2))
    if achado.group(3):
        ano = int(achado.group(3))
        ano = ano + 2000 if ano < 100 else ano
    else:
        ano = ano_padrao or datetime.now().year

    try:
        return date(ano, mes, dia)
    except ValueError:
        return None


def parse_previsto(valor) -> timedelta | None:
    """'07:00-09:00 09:30-12:00' -> soma dos intervalos."""
    if vazio(valor):
        return None
    total = timedelta(0)
    achou = False
    for inicio, fim in RE_INTERVALO.findall(str(valor)):
        h1, m1 = (int(x) for x in inicio.split(":"))
        h2, m2 = (int(x) for x in fim.split(":"))
        inicio_td = timedelta(hours=h1, minutes=m1)
        fim_td = timedelta(hours=h2, minutes=m2)
        if fim_td < inicio_td:
            fim_td += timedelta(days=1)
        total += fim_td - inicio_td
        achou = True
    return total if achou else None


def fmt_hora(valor) -> str:
    if vazio(valor):
        return ""
    if isinstance(valor, time):
        return valor.strftime("%H:%M")
    return str(valor)[:5]


def fmt_duracao(valor) -> str:
    if vazio(valor):
        return ""
    if isinstance(valor, str):
        return valor
    segundos = int(valor.total_seconds())
    sinal = "-" if segundos < 0 else ""
    segundos = abs(segundos)
    horas, resto = divmod(segundos, 3600)
    return f"{sinal}{horas:02d}:{resto // 60:02d}"

In [6]:
# =====================================================================
# CÉLULA 6 — Leitura do arquivo
# =====================================================================
def ler_relatorio(caminho: Path | str) -> tuple[pd.DataFrame, dict]:
    caminho = Path(caminho)
    melhor = None

    for encoding in ENCODINGS_TENTATIVA:
        for separador in SEPARADORES_TENTATIVA:
            try:
                df = pd.read_csv(caminho, encoding=encoding, sep=separador,
                                 dtype=str, keep_default_na=False,
                                 na_values=[""])
            except (UnicodeDecodeError, pd.errors.ParserError, ValueError):
                continue

            if df.shape[1] >= 5 and (melhor is None
                                     or df.shape[1] > melhor[0].shape[1]):
                melhor = (df, encoding, separador)

        if melhor is not None:
            break

    if melhor is None:
        raise ValueError(f"Nenhum leiaute reconhecido em {caminho.name}")

    df, encoding, separador = melhor
    df.columns = [str(c).strip() for c in df.columns]
    df = df.dropna(how="all").reset_index(drop=True)

    info = {
        "arquivo": caminho.name,
        "encoding": encoding,
        "separador": separador,
        "linhas": len(df),
        "colunas": len(df.columns),
    }
    return df, info

In [7]:
# =====================================================================
# CÉLULA 7 — Normalização: uma linha por colaborador por dia
# =====================================================================
def _somar(inicio: time | None, fim: time | None) -> timedelta | None:
    """fim - inicio em timedelta, tratando virada de meia-noite."""
    if not isinstance(inicio, time) or not isinstance(fim, time):
        return None
    t0 = timedelta(hours=inicio.hour, minutes=inicio.minute)
    t1 = timedelta(hours=fim.hour, minutes=fim.minute)
    if t1 < t0:
        t1 += timedelta(days=1)
    return t1 - t0


def calcular_blocos(registro: dict, pares) -> dict[int, timedelta | None]:
    """Tempo efetivamente trabalhado em cada bloco (entrada_n -> saída_n)."""
    return {
        n: _somar(registro.get(f"entrada_{n}"), registro.get(f"saida_{n}"))
        for n in sorted(pares)
    }


def calcular_intervalos(registro: dict, pares) -> dict[int, timedelta | None]:
    """Intervalo NÃO remunerado após cada par (saída_n -> entrada_{n+1})."""
    numeros = sorted(pares)
    return {
        atual: _somar(registro.get(f"saida_{atual}"),
                      registro.get(f"entrada_{proximo}"))
        for atual, proximo in zip(numeros, numeros[1:])
    }


def rotulo_intervalo(n: int) -> str:
    return ROTULO_INTERVALO.get(n, f"Intervalo {n}")


def calcular_horas_trabalhadas(blocos: dict[int, timedelta | None]) -> timedelta:
    """Soma só os blocos trabalhados — café e almoço ficam fora."""
    return sum((v for v in blocos.values() if v), timedelta(0))


def calcular_horas_brutas(registro: dict, pares) -> timedelta | None:
    """Primeira entrada até a última saída, incluindo os intervalos."""
    numeros = sorted(pares)
    if not numeros:
        return None

    entradas = [registro.get(f"entrada_{n}") for n in numeros]
    saidas = [registro.get(f"saida_{n}") for n in numeros]
    entradas = [h for h in entradas if isinstance(h, time)]
    saidas = [h for h in saidas if isinstance(h, time)]

    if not entradas or not saidas:
        return None

    return _somar(min(entradas), max(saidas))


def classificar_status(registro: dict, pares) -> str:
    pedacos = [texto(registro.get("justificativas"))]
    for n in pares:
        pedacos.append(texto(registro.get(f"entrada_{n}")))
        pedacos.append(texto(registro.get(f"saida_{n}")))
    tudo = " ".join(pedacos).lower()

    if "feriado" in tudo:
        return "feriado"
    if any(p in tudo for p in ("folga", "descanso", "dsr")):
        return "folga"
    if any(p in tudo for p in ("afastamento", "licença", "licenca", "inss")):
        return "afastamento"
    if "falta" in tudo:
        return "falta"
    if any(p in tudo for p in ("justific", "abon", "atestado", "férias", "ferias")):
        return "justificado"

    marcacoes = sum(
        1 for n in pares
        for lado in ("entrada", "saida")
        if registro.get(f"{lado}_{n}") is not None
    )

    if marcacoes == 0:
        return "falta" if registro.get("previsto_horas") else "folga"
    if marcacoes % 2:
        return "incompleto"
    return "ok"


def normalizar(caminho: Path | str, ano_padrao: int | None = None) -> dict:
    bruto, info = ler_relatorio(caminho)
    campos, pares, problemas = conferir_leiaute(bruto)

    if problemas:
        raise ValueError("; ".join(problemas))

    linhas = []
    for _, row in bruto.iterrows():
        original = row.to_dict()

        registro = {
            "nome": texto(original.get(campos["nome"])),
            "cpf": somente_digitos(original.get(campos["cpf"])),
            "dia_texto": texto(original.get(campos["dia"])),
            "previsto": texto(original.get(campos.get("previsto"))),
            "justificativas": texto(original.get(campos.get("justificativas"))),
        }

        registro["data"] = parse_dia(registro["dia_texto"], ano_padrao)
        registro["previsto_horas"] = parse_previsto(registro["previsto"])
        registro["teve_previsto"] = bool(registro["previsto_horas"])

        for n, colunas in pares.items():
            registro[f"entrada_{n}"] = parse_hora(
                original.get(colunas.get("entrada")))
            registro[f"saida_{n}"] = parse_hora(
                original.get(colunas.get("saida")))

        for campo in CAMPOS_DURACAO:
            origem = campos.get(campo)
            registro[campo] = parse_duracao(
                original.get(origem)) if origem else None

        registro["qtd_marcacoes"] = sum(
            1 for n in pares for lado in ("entrada", "saida")
            if registro.get(f"{lado}_{n}") is not None
        )

        # ---- horas reais: café e almoço excluídos ----
        blocos = calcular_blocos(registro, pares)
        intervalos = calcular_intervalos(registro, pares)

        registro["blocos"] = blocos
        registro["intervalos"] = intervalos

        for n in sorted(pares):
            registro[f"bloco_{n}"] = blocos.get(n)
        for n in sorted(intervalos):
            registro[f"intervalo_{n}"] = intervalos[n]

        registro["horas_trabalhadas"] = calcular_horas_trabalhadas(blocos)
        registro["horas_brutas"] = calcular_horas_brutas(registro, pares)
        registro["total_intervalos"] = sum(
            (v for v in intervalos.values() if v), timedelta(0))
        registro["qtd_intervalos"] = sum(
            1 for v in intervalos.values() if v)

        registro["status"] = classificar_status(registro, pares)

        if registro["total_normais"] is not None:
            registro["divergencia"] = (registro["horas_trabalhadas"]
                                       - registro["total_normais"])
        else:
            registro["divergencia"] = None

        linhas.append(registro)

    df = pd.DataFrame(linhas)
    df["cpf_formatado"] = df["cpf"].apply(formatar_cpf)
    df["dia_semana"] = df["data"].apply(
        lambda d: DIAS_ABREV[d.weekday()] if isinstance(d, date) else "")

    return {"arquivo": Path(caminho), "info": info, "campos": campos,
            "pares": pares,
            "df": df.sort_values(["nome", "data"]).reset_index(drop=True)}

In [8]:
# =====================================================================
# CÉLULA 8 — Validações
# =====================================================================
SITUACOES_SEM_MARCACAO = ("folga", "feriado", "sem registro")


def _verificar_ordem(registro: dict, pares) -> list[str]:
    """Ordem cronológica DENTRO de cada par e ENTRE pares consecutivos."""
    avisos = []
    numeros = sorted(pares)

    for n in numeros:
        entrada = registro.get(f"entrada_{n}")
        saida = registro.get(f"saida_{n}")
        if isinstance(entrada, time) and isinstance(saida, time) and saida < entrada:
            avisos.append(f"saída {n} anterior à entrada {n} (virada de dia?)")

    for atual, proximo in zip(numeros, numeros[1:]):
        saida_atual = registro.get(f"saida_{atual}")
        entrada_prox = registro.get(f"entrada_{proximo}")
        if (isinstance(saida_atual, time) and isinstance(entrada_prox, time)
                and entrada_prox < saida_atual):
            avisos.append(f"entrada {proximo} anterior à saída {atual}")

    return avisos


def validar(resultado: dict) -> list[str]:
    df = resultado["df"]
    pares = resultado["pares"]
    problemas = []

    for _, r in df[df["data"].isna()].iterrows():
        problemas.append(f"Dia ilegível: '{r['dia_texto']}' ({r['nome']})")

    for _, r in df[df["cpf"] == ""].iterrows():
        problemas.append(
            f"Linha sem CPF: {r['nome']} — dia '{r['dia_texto']}'")

    invalidos = df[(df["cpf"] != "") & (~df["cpf"].map(cpf_valido))]
    for cpf in sorted(invalidos["cpf"].unique()):
        problemas.append(
            f"CPF com dígito verificador inválido: {mascarar_cpf(cpf)}")

    mesmo_cpf = df.groupby("cpf")["nome"].nunique()
    for cpf in mesmo_cpf[mesmo_cpf > 1].index:
        problemas.append(
            f"CPF {mascarar_cpf(cpf)} aparece com nomes diferentes")

    repetido = df[df.duplicated(
        ["cpf", "data"], keep=False) & df["data"].notna()]
    for _, r in repetido.iterrows():
        problemas.append(f"Dia repetido: {r['nome']} em {r['data']:%d/%m/%Y}")

    for _, r in df[df["status"] == "incompleto"].iterrows():
        problemas.append(
            f"Marcação sem par: {r['nome']} em {r['data']:%d/%m/%Y} "
            f"({r['qtd_marcacoes']} marcas)")

    for _, r in df[df["status"] == "falta"].iterrows():
        problemas.append(
            f"Dia previsto sem marcação: {r['nome']} em {r['data']:%d/%m/%Y}")

    for _, r in df[~df["status"].isin(SITUACOES_SEM_MARCACAO)].iterrows():
        for aviso in _verificar_ordem(r, pares):
            problemas.append(
                f"{aviso}: {r['nome']} em {r['data']:%d/%m/%Y}")

    return problemas


def imprimir_resumo(resultado: dict) -> None:
    df = resultado["df"]
    info = resultado["info"]
    datas = df["data"].dropna()

    print("=" * 74)
    print("RELATÓRIO ANALÍTICO — RHiD")
    print("=" * 74)
    print(f"arquivo           : {info['arquivo']}")
    print(
        f"encoding/separador: {info['encoding']} / {repr(info['separador'])}")
    print(f"linhas / colunas  : {info['linhas']} / {info['colunas']}")
    print(f"pares detectados  : {sorted(resultado['pares'])}")
    print(f"colaboradores     : {df['cpf'].nunique()}")
    if not datas.empty:
        print(
            f"período           : {datas.min():%d/%m/%Y} a {datas.max():%d/%m/%Y}")
    print(f"dias              : {len(df)}")

    print("-" * 74)
    print("situação dos dias:")
    for status, qtd in df["status"].value_counts().items():
        print(f"  {status:14}: {qtd}")

    previstos_sem_marca = df[df["teve_previsto"] & (df["qtd_marcacoes"] == 0)]
    if not previstos_sem_marca.empty:
        print("-" * 74)
        print(
            f"dias previstos sem nenhuma marcação: {len(previstos_sem_marca)}")

    problemas = validar(resultado)
    print("-" * 74)
    if problemas:
        print(f"{len(problemas)} inconsistência(s):")
        for p in problemas[:25]:
            print("  •", p)
        if len(problemas) > 25:
            print(f"  ... e mais {len(problemas) - 25}.")
    else:
        print("Nenhuma inconsistência encontrada.")
    print("=" * 74)

In [9]:
# =====================================================================
# CÉLULA 9 — Uma linha por marcação
# =====================================================================
def montar_marcacoes(df: pd.DataFrame,
                     pares: dict[int, dict[str, str]],
                     mascarar: bool = False) -> pd.DataFrame:
    linhas = []

    for _, r in df.iterrows():
        for ordem, n in enumerate(sorted(pares), start=1):
            for lado, rotulo in (("entrada", f"Entrada {n}"), ("saida", f"Saída {n}")):
                hora = r.get(f"{lado}_{n}")
                if not isinstance(hora, time):
                    continue
                linhas.append({
                    "Colaborador": r["nome"],
                    "CPF": mascarar_cpf(r["cpf"]) if mascarar else r["cpf_formatado"],
                    "Data": r["data"],
                    "Dia": r["dia_semana"],
                    "Marcação": rotulo,
                    "Hora": hora.strftime("%H:%M"),
                    "Ordem": ordem,
                    "Tipo": lado,
                    "Situação do dia": r["status"],
                })

    if not linhas:
        return pd.DataFrame()

    return (pd.DataFrame(linhas)
            .sort_values(["Colaborador", "Data", "Hora"])
            .reset_index(drop=True))

In [10]:
# =====================================================================
# CÉLULA 10 — Resumo por colaborador
# =====================================================================
def _somar_tempo(serie) -> timedelta:
    """Soma valores de duração ignorando None e NaN."""
    total = timedelta(0)
    for valor in serie:
        if isinstance(valor, timedelta):
            total += valor
    return total


def resumo_por_colaborador(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    linhas = []
    for (cpf, nome), sub in df.groupby(["cpf", "nome"], dropna=False):
        saldo = sub.sort_values("data")["banco_saldo"].dropna()

        cafe = sub.get("intervalo_1", pd.Series(dtype=object))
        almoco = sub.get("intervalo_2", pd.Series(dtype=object))

        linhas.append({
            "Colaborador": nome or formatar_cpf(cpf),
            "CPF": formatar_cpf(cpf),
            "Dias": sub["data"].nunique(),
            "Marcações": int(sub["qtd_marcacoes"].sum()),
            "Faltas": int((sub["status"] == "falta").sum()),
            "Folgas": int((sub["status"] == "folga").sum()),
            "Feriados": int((sub["status"] == "feriado").sum()),
            "Justificados": int((sub["status"] == "justificado").sum()),
            "Incompletos": int((sub["status"] == "incompleto").sum()),
            "Previsto": fmt_duracao(_somar_tempo(sub["previsto_horas"])),
            "Trabalhadas": fmt_duracao(_somar_tempo(sub["horas_trabalhadas"])),
            "Bruto": fmt_duracao(_somar_tempo(sub["horas_brutas"])),
            "Café": fmt_duracao(_somar_tempo(cafe)),
            "Almoço": fmt_duracao(_somar_tempo(almoco)),
            "Normais": fmt_duracao(_somar_tempo(sub["total_normais"])),
            "Noturno": fmt_duracao(_somar_tempo(sub["total_noturno"])),
            "Extra Diurna": fmt_duracao(_somar_tempo(sub["extra_diurna"])),
            "Extra Noturna": fmt_duracao(_somar_tempo(sub["extra_noturna"])),
            "Falta/Atraso": fmt_duracao(_somar_tempo(sub["falta_e_atraso"])),
            "Saldo final": fmt_duracao(saldo.iloc[-1]) if len(saldo) else "",
        })

    return pd.DataFrame(linhas).sort_values("Colaborador").reset_index(drop=True)

In [11]:
# =====================================================================
# CÉLULA 11 — Exportação: RESUMO | MARCACOES | uma aba por colaborador
# =====================================================================
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

ABAS_FIXAS = ("RESUMO", "MARCACOES")
LIMITE_ABA = 31
CARACTERES_PROIBIDOS = r"[:\\/?*\[\]]"

LINHA_TITULO = 1
LINHA_SUBTITULO = 2
LINHA_CABECALHO = 4

COR_CABECALHO = "1F3B5B"
COR_FALTA = "FBE4E4"
COR_FOLGA = "F0F2F5"
COR_DIVERGENCIA = "FDF2E3"
COR_TRABALHADA = "E8F0E8"

CORES_SITUACAO = {
    "falta": COR_FALTA,
    "incompleto": COR_DIVERGENCIA,
    "justificado": COR_DIVERGENCIA,
    "folga": COR_FOLGA,
    "feriado": COR_FOLGA,
}

LARGURAS_DIAS = {
    "Data": 11, "Dia": 6, "Previsto": 32,
    "Café": 9, "Almoço": 9,
    "Trabalhadas": 12, "Bruto": 9,
    "Normais": 9, "Noturno": 9, "Falta/Atraso": 13,
    "Extra Diurna": 13, "Extra Noturna": 13,
    "Banco dia": 10, "Saldo": 10,
    "Situação": 14, "Justificativas": 44,
}

LARGURAS_RESUMO = {
    "Colaborador": 36, "CPF": 18, "Dias": 8, "Marcações": 11,
    "Faltas": 8, "Folgas": 8, "Feriados": 9, "Justificados": 12,
    "Incompletos": 12, "Previsto": 10, "Trabalhadas": 12, "Bruto": 9,
    "Café": 9, "Almoço": 9, "Normais": 9, "Noturno": 9,
    "Extra Diurna": 13, "Extra Noturna": 13, "Falta/Atraso": 13,
    "Saldo final": 12,
}

LARGURAS_MARCACOES = {
    "Colaborador": 36, "CPF": 18, "Data": 11, "Dia": 6,
    "Marcação": 12, "Hora": 8, "Ordem": 7, "Tipo": 9,
    "Situação do dia": 16,
}


def nome_aba_seguro(nome: str, usados: set[str]) -> str:
    base = re.sub(CARACTERES_PROIBIDOS, "-", texto(nome)).strip()
    base = re.sub(r"\s+", " ", base) or "SEM NOME"
    base = base[:LIMITE_ABA]

    candidato, i = base, 2
    while candidato.upper() in usados:
        sufixo = f" ({i})"
        candidato = base[:LIMITE_ABA - len(sufixo)] + sufixo
        i += 1

    usados.add(candidato.upper())
    return candidato


def titulo_mesclado(ws, linha: int, colunas: int, valor: str,
                    tamanho: float = 14, negrito: bool = True,
                    italico: bool = False, altura: float | None = None) -> None:
    colunas = max(1, colunas)

    ancora = ws.cell(row=linha, column=1)
    ancora.value = valor
    ancora.font = Font(bold=negrito, size=tamanho, italic=italico)
    ancora.alignment = Alignment(horizontal="left", vertical="center")

    if colunas > 1:
        ws.merge_cells(start_row=linha, start_column=1,
                       end_row=linha, end_column=colunas)
        ancora = ws.cell(row=linha, column=1)
        ancora.alignment = Alignment(horizontal="left", vertical="center")

    if altura:
        ws.row_dimensions[linha].height = altura


def larguras_por_coluna(colunas, mapa: dict, padrao: int = 9) -> list[int]:
    larguras = []
    for nome in colunas:
        nome = str(nome)
        if nome in mapa:
            larguras.append(mapa[nome])
        elif nome.startswith(("Entrada", "Saída", "Saida")):
            larguras.append(9)
        else:
            larguras.append(padrao)
    return larguras


def montar_tabela_dias(sub: pd.DataFrame, pares) -> pd.DataFrame:
    linhas = []
    for _, r in sub.sort_values("data").iterrows():
        linha = {
            "Data": r["data"],
            "Dia": r["dia_semana"],
            "Previsto": r["previsto"],
        }
        for n in sorted(pares):
            linha[f"Entrada {n}"] = fmt_hora(r.get(f"entrada_{n}"))
            linha[f"Saída {n}"] = fmt_hora(r.get(f"saida_{n}"))

        numeros = sorted(pares)
        for atual in numeros[:-1]:
            linha[rotulo_intervalo(atual)] = fmt_duracao(
                r.get(f"intervalo_{atual}"))

        linha["Trabalhadas"] = fmt_duracao(r.get("horas_trabalhadas"))
        linha["Bruto"] = fmt_duracao(r.get("horas_brutas"))
        linha["Normais"] = fmt_duracao(r.get("total_normais"))
        linha["Noturno"] = fmt_duracao(r.get("total_noturno"))
        linha["Falta/Atraso"] = fmt_duracao(r.get("falta_e_atraso"))
        linha["Extra Diurna"] = fmt_duracao(r.get("extra_diurna"))
        linha["Extra Noturna"] = fmt_duracao(r.get("extra_noturna"))
        linha["Banco dia"] = fmt_duracao(r.get("banco_total"))
        linha["Saldo"] = fmt_duracao(r.get("banco_saldo"))
        linha["Situação"] = r["status"]
        linha["Justificativas"] = r["justificativas"]
        linhas.append(linha)

    return pd.DataFrame(linhas)


def _estilizar(ws, linha_cabecalho: int, colunas: int, ultima_linha: int,
               larguras: list[int], cores: dict[int, str] | None = None,
               colunas_destaque: tuple[int, ...] = ()):
    for indice, largura in enumerate(larguras[:colunas], start=1):
        ws.column_dimensions[get_column_letter(indice)].width = largura

    for coluna in range(1, colunas + 1):
        celula = ws.cell(row=linha_cabecalho, column=coluna)
        celula.font = Font(bold=True, color="FFFFFF")
        celula.fill = PatternFill("solid", fgColor=COR_CABECALHO)
        celula.alignment = Alignment(horizontal="center", vertical="center")

    if ultima_linha >= linha_cabecalho:
        ws.auto_filter.ref = (
            f"A{linha_cabecalho}:{get_column_letter(colunas)}{ultima_linha}")

    ws.freeze_panes = ws.cell(row=linha_cabecalho + 1, column=1)

    for linha in range(linha_cabecalho + 1, ultima_linha + 1):
        cor = (cores or {}).get(linha)
        if cor:
            for coluna in range(1, colunas + 1):
                ws.cell(row=linha, column=coluna).fill = PatternFill(
                    "solid", fgColor=cor)

    for coluna in colunas_destaque:
        if coluna > colunas:
            continue
        for linha in range(linha_cabecalho + 1, ultima_linha + 1):
            ws.cell(row=linha, column=coluna).font = Font(bold=True)


def exportar(resultado: dict, destino: Path | str | None = None) -> Path:
    df = resultado["df"]
    pares = resultado["pares"]
    destino = Path(destino or resultado["arquivo"].with_name(
        f"{resultado['arquivo'].stem}_tratado.xlsx"))

    resumo = resumo_por_colaborador(df)
    marcacoes = montar_marcacoes(df, pares)
    usados = {nome.upper() for nome in ABAS_FIXAS}

    with pd.ExcelWriter(destino, engine="openpyxl") as writer:
        # ---------------- RESUMO ----------------
        ws = writer.book.create_sheet("RESUMO")
        colunas = max(1, len(resumo.columns))

        if not resumo.empty:
            resumo.to_excel(writer, sheet_name="RESUMO",
                            index=False, startrow=LINHA_CABECALHO - 1)

        titulo_mesclado(ws, LINHA_TITULO, colunas,
                        "Resumo por colaborador", 14, altura=22)
        titulo_mesclado(ws, LINHA_SUBTITULO, colunas,
                        f"{resultado['arquivo'].name} · "
                        f"gerado em {datetime.now():%d/%m/%Y %H:%M}",
                        10, negrito=False, italico=True, altura=16)

        if not resumo.empty:
            destaque = ()
            if "Trabalhadas" in resumo.columns:
                destaque = (list(resumo.columns).index("Trabalhadas") + 1,)
            _estilizar(ws, LINHA_CABECALHO, colunas, LINHA_CABECALHO + len(resumo),
                       larguras_por_coluna(resumo.columns, LARGURAS_RESUMO),
                       colunas_destaque=destaque)

        # ---------------- MARCACOES ----------------
        ws = writer.book.create_sheet("MARCACOES")
        colunas = max(1, len(marcacoes.columns))

        if not marcacoes.empty:
            marcacoes.to_excel(writer, sheet_name="MARCACOES",
                               index=False, startrow=LINHA_CABECALHO - 1)

        titulo_mesclado(ws, LINHA_TITULO, colunas,
                        "Todas as marcações", 14, altura=22)
        titulo_mesclado(ws, LINHA_SUBTITULO, colunas,
                        f"{len(marcacoes)} marcação(ões)",
                        10, negrito=False, italico=True, altura=16)

        if not marcacoes.empty:
            _estilizar(ws, LINHA_CABECALHO, colunas,
                       LINHA_CABECALHO + len(marcacoes),
                       larguras_por_coluna(marcacoes.columns, LARGURAS_MARCACOES))

        # ---------------- UMA ABA POR COLABORADOR ----------------
        for (cpf, nome), sub in df.groupby(["cpf", "nome"], dropna=False):
            aba = nome_aba_seguro(nome or formatar_cpf(cpf), usados)
            tabela = montar_tabela_dias(sub, pares)
            colunas = max(1, len(tabela.columns))

            saldo_final = sub.sort_values("data")["banco_saldo"].dropna()
            total_trab = fmt_duracao(_somar_tempo(sub["horas_trabalhadas"]))

            ws = writer.book.create_sheet(aba)

            if not tabela.empty:
                tabela.to_excel(writer, sheet_name=aba,
                                index=False, startrow=LINHA_CABECALHO - 1)

            titulo_mesclado(ws, LINHA_TITULO, colunas,
                            nome or f"CPF {formatar_cpf(cpf)}", 14, altura=22)

            titulo_mesclado(
                ws, LINHA_SUBTITULO, colunas,
                f"CPF {formatar_cpf(cpf)}   ·   "
                f"{sub['data'].nunique()} dia(s)   ·   "
                f"trabalhadas {total_trab}   ·   "
                f"saldo final "
                f"{fmt_duracao(saldo_final.iloc[-1]) if len(saldo_final) else '—'}",
                10, negrito=False, italico=True, altura=16)

            if tabela.empty:
                continue

            cores = {}
            for indice, situacao in enumerate(tabela["Situação"],
                                              start=LINHA_CABECALHO + 1):
                cor = CORES_SITUACAO.get(situacao)
                if cor:
                    cores[indice] = cor

            destaque = ()
            if "Trabalhadas" in tabela.columns:
                destaque = (list(tabela.columns).index("Trabalhadas") + 1,)

            _estilizar(ws, LINHA_CABECALHO, colunas,
                       LINHA_CABECALHO + len(tabela),
                       larguras_por_coluna(tabela.columns, LARGURAS_DIAS),
                       cores, destaque)

        if "Sheet" in writer.book.sheetnames:
            del writer.book["Sheet"]

    return destino

In [12]:
# =====================================================================
# CÉLULA 12 — Pipeline: lê, valida e exporta
# =====================================================================
def processar(caminho: Path, exportar_excel: bool = True) -> dict:
    resultado = normalizar(caminho)
    imprimir_resumo(resultado)

    resultado["marcacoes"] = montar_marcacoes(
        resultado["df"], resultado["pares"])
    resultado["resumo"] = resumo_por_colaborador(resultado["df"])

    if exportar_excel:
        destino = exportar(resultado)
        print(f"\nPlanilha: {destino.name} "
              f"({destino.stat().st_size / 1024:.1f} KB)")

        wb = load_workbook(destino, read_only=True)
        try:
            print(f"{len(wb.sheetnames)} abas:")
            for nome_aba in wb.sheetnames:
                print(f"  • {nome_aba}")
        finally:
            wb.close()

    return resultado


RESULTADOS = [processar(arquivo) for arquivo in ARQUIVOS]

if RESULTADOS:
    R = RESULTADOS[-1]
    DF_DIAS = R["df"]
    DF_MARCACOES = R["marcacoes"]
    DF_RESUMO = R["resumo"]

    print("\nResumo por colaborador:")
    print(DF_RESUMO.to_string(index=False))
else:
    print("Nenhum arquivo encontrado em", PASTA_ARQUIVOS)

RELATÓRIO ANALÍTICO — RHiD
arquivo           : relatorio_20261026_1501.CSV
encoding/separador: utf-8-sig / ';'
linhas / colunas  : 330 / 22
pares detectados  : [1, 2, 3]
colaboradores     : 11
período           : 26/09/2026 a 25/10/2026
dias              : 330
--------------------------------------------------------------------------
situação dos dias:
  ok            : 196
  justificado   : 67
  folga         : 49
  feriado       : 10
  falta         : 7
  incompleto    : 1
--------------------------------------------------------------------------
dias previstos sem nenhuma marcação: 54
--------------------------------------------------------------------------
8 inconsistência(s):
  • Marcação sem par: ROQUE CARLOS NOGUEIRA JUNIOR em 21/10/2026 (3 marcas)
  • Dia previsto sem marcação: DIRCEU BENEDITO em 05/10/2026
  • Dia previsto sem marcação: IORAN RAONE DE ALMEIDA GOIS em 29/09/2026
  • Dia previsto sem marcação: IORAN RAONE DE ALMEIDA GOIS em 06/10/2026
  • Dia previsto sem marca